# ResNet 18 Transfer Learning

### Objective

Train an ImageNet-pretrained ResNet18 model to classify a subset of NIH's chest X-rays into Effusion (1) or No Finding (0). 

This experiment investigates whether transfer learning improves classification performance compared with CNN trained from scratch in notebook 04_scratch_cnn.ipynb.

## Training Plan

Training will take place in 2 stages:
*1. Feature Extraction:* Freeze the pretrained ResNet18 backbone and train only the new classification head.
*2. Fine-tuning:* Unfreeze the backbone and continue training with a lower learning rate for the pretrained layers.

Both stages will use the existing preprocessing pipeline and patient-level dataset splits. The model will use "BCEWithLogitsLoss" with positive class weighting to account for class imbalance. 

The best model checkpoint will be selected based on validation ROC-AUC. The held-out test set will remain untouched until final evaluation in notebook 06_scratch_vs_resnet18_evaluation.ipynb.

## 01. Set Up Notebook

In [1]:
# Standard library
import random
import sys
from pathlib import Path

# Data and visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models

# Project source code
cwd = Path.cwd().resolve()
PROJECT_DIR = next((path for path in [cwd, *cwd.parents]
                    if path.name == "transfer-learning-resnet"
                    and (path / "src" / "data_loader.py").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate project directory.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))    #Allow imports from project's src folder

# Paths
DATA_DIR = PROJECT_DIR.parent / "data"
IMAGE_DIR = DATA_DIR / "raw" / "images"
CSV_PATH = DATA_DIR / "processed" / "mlp_dataset.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Project directory:", PROJECT_DIR)  
print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)          

Project directory: /Users/hweeenheah/Downloads/EDA_Resources/nih-chestxray-ml/transfer-learning-resnet
Device: mps
PyTorch version: 2.15.0
Torchvision version: 0.29.2


## 02. Load existing datasets and DataLoaders

In [2]:
from src.data_loader import ChestXrayDataset
from src.transforms import get_preprocessing_transform

BATCH_SIZE = 32
NUM_WORKERS = 0

preprocess_transform = get_preprocessing_transform()

train_dataset = ChestXrayDataset(CSV_PATH, IMAGE_DIR, preprocess_transform, split="train")
val_dataset = ChestXrayDataset(CSV_PATH, IMAGE_DIR, preprocess_transform, split="val")
test_dataset = ChestXrayDataset(CSV_PATH, IMAGE_DIR, preprocess_transform, split="test")

# Reproducible training shuffle
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# Verify data pipeline
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

images, labels, patient_ids = next(iter(train_loader))
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Patient IDs:", patient_ids.shape)
print("Image dtype:", images.dtype)

Train: 2208
Validation: 554
Test: 479
Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])
Patient IDs: torch.Size([32])
Image dtype: torch.float32


## 03. Initialise pretrained ResNet18 model

Note: Original classifier replaced with single-output layer. Backbone frozen.

In [3]:
from torchvision.models import resnet18, ResNet18_Weights

# Load ImageNet-pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace original 1000-class classifier with a single-output layer
model.fc = nn.Linear(model.fc.in_features, 1)

# Freeze all pretrained parameters
for param in model.parameters():
    param.requires_grad = False

# Allow new classification head to be trained
for param in model.fc.parameters():
    param.requires_grad = True

# Move model to selected device
model = model.to(DEVICE)

# Check trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

Trainable parameters: 513
Total parameters: 11,177,025


## 04. Stage 1 Training

Stage 1: Freeze the backbone, replace the final layer with a single-output classifier, and train only the new head for a few epochs. 

We will use the same training-derived positive class weight as the scratch CNN.

### 4.1 Configure Loss Function and Optimiser

In [4]:
# Loss function with positive class weighting
pos_weight = torch.tensor([1939 / 269], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimiser: only update classification head
LEARNING_RATE = 0.001
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE)

# Stage 1 training duration
STAGE1_EPOCHS = 3

print(f"Positive class weight: {pos_weight.item():.4f}")

Positive class weight: 7.2082


### 4.2 Training and Validation functions

In [5]:
from sklearn.metrics import roc_auc_score

# Training function
def train_one_epoch(model, loader, criterion, optimizer, device,
                    freeze_backbone_bn=False):
    model.train()
    # Freeze BatchNorm too
    if freeze_backbone_bn:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
    running_loss = 0.0
    for images, labels, _ in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

# Validation function
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_probabilities = []
    with torch.no_grad():    #prevents gradient calculations during validation
        for images, labels, _ in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            probabilities = torch.sigmoid(outputs)
            running_loss += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy().ravel())
            all_probabilities.extend(probabilities.cpu().numpy().ravel())
    val_loss = running_loss / len(loader.dataset)
    val_auc = roc_auc_score(all_labels, all_probabilities)
    return val_loss, val_auc        

### 4.3 Initialise Checkpoint Tracking

To retain best Stage 1 checkpoint

In [6]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []}

best_val_auc = 0.0
stage1_best_model_state = None

print("Training history and checkpoint tracking initialised.")

Training history and checkpoint tracking initialised.


In [7]:
"""
This cell resets the model to ensure reproducibility
"""

# Reset random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# Reset training DataLoader shuffle order
train_loader.generator.manual_seed(SEED)
# Reinitialise pretrained ResNet18 and classification head
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 1)

# Freeze backbone; train classification head only
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

# Reinitialise optimiser
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE
)

# Reset checkpoint tracking and history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

best_val_auc = 0.0
stage1_best_model_state = None

print("Stage 1 reset complete.")


Stage 1 reset complete.


### 4.4 Train stage 1

Runs 3 epochs with backbone frozen

In [8]:
for epoch in range(STAGE1_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, freeze_backbone_bn=True)
    val_loss, val_auc = validate_one_epoch(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)
    # Keep checkpoint with best val_auc
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        stage1_best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()}
    # Results
    print(
        f"Epoch {epoch + 1}/{STAGE1_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}")
print(f"\nBest Stage 1 validation AUC: {best_val_auc:.4f}")        

Epoch 1/3 | Train Loss: 1.1076 | Val Loss: 1.0013 | Val AUC: 0.8227
Epoch 2/3 | Train Loss: 0.9897 | Val Loss: 0.9353 | Val AUC: 0.8302
Epoch 3/3 | Train Loss: 0.9116 | Val Loss: 0.9147 | Val AUC: 0.8345

Best Stage 1 validation AUC: 0.8345


### 4.5 Restore best Stage 1 checkpoint

In [9]:
model.load_state_dict(stage1_best_model_state)
print(f"Restored best Stage 1 model (Validation AUC: {best_val_auc:.4f})")

Restored best Stage 1 model (Validation AUC: 0.8345)


## 05. Stage 2 Training

Stage 2: Unfreeze the backbone, use differential learning rates (smaller lr for pretrained backbone to preserve learned features, larger lr for classification head to learn from scratch quickly) and train for a few epochs. 

We will use the same training-derived positive class weight as the scratch CNN.

### 5.1 Unfreeze backbone and separate learning rates

In [10]:
# Unfreeze entire model
for param in model.parameters():
    param.requires_grad = True

# Separate learning rates for backbone and classification head
BACKBONE_LR = 0.0001
HEAD_LR = 0.001

optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters()},
    {"params": model.layer2.parameters()},
    {"params": model.layer3.parameters()},
    {"params": model.layer4.parameters()},
    {"params": model.conv1.parameters()},
    {"params": model.bn1.parameters()},
    {"params": model.fc.parameters(), "lr": HEAD_LR}],
    lr = BACKBONE_LR)

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 11,177,025


### 5.2 Initialise checkpoint tracking

To retain best stage 2 checkpoint while preserving history (stage 1 results preserved, stage 2 results will be appended to it). 

In [11]:
"""
This cell resets the model to ensure reproducibility
"""
# Reset random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)    

# Reset Stage 2 to the best Stage 1 checkpoint
model.load_state_dict(stage1_best_model_state)

# Recreate optimiser to reset Adam's internal state
optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters()},
    {"params": model.layer2.parameters()},
    {"params": model.layer3.parameters()},
    {"params": model.layer4.parameters()},
    {"params": model.conv1.parameters()},
    {"params": model.bn1.parameters()},
    {"params": model.fc.parameters(), "lr": HEAD_LR}
], lr=BACKBONE_LR)

# Reset training DataLoader shuffle sequence
train_loader.generator.manual_seed(SEED)

# Reset Stage 2 tracking
best_val_auc = 0.0
stage2_best_model_state = None
stage2_best_epoch = None

print("Stage 2 reset complete.")

Stage 2 reset complete.


### 5.3 Train stage 2

Run 10 epochs with unfrozen backbone. Loss function will remain the same (BCEWithLogitsLoss does not depend on whether backbone is frozen or unfrozen. The same loss and pos_weight applies to both stages.)

In [12]:
# Stage 2 training duration
STAGE2_EPOCHS = 10

for epoch in range(STAGE2_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_auc = validate_one_epoch(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        stage2_best_epoch = epoch + 1
        stage2_best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()}
    print(
        f"Stage 2 Epoch {epoch + 1}/{STAGE2_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}") 
print(f"\nBest Stage 2 validation AUC: {best_val_auc:.4f}")           

Stage 2 Epoch 1/10 | Train Loss: 0.9281 | Val Loss: 0.9182 | Val AUC: 0.8547
Stage 2 Epoch 2/10 | Train Loss: 0.3198 | Val Loss: 1.6188 | Val AUC: 0.8560
Stage 2 Epoch 3/10 | Train Loss: 0.1149 | Val Loss: 1.7700 | Val AUC: 0.8812
Stage 2 Epoch 4/10 | Train Loss: 0.0527 | Val Loss: 1.6209 | Val AUC: 0.8620
Stage 2 Epoch 5/10 | Train Loss: 0.0218 | Val Loss: 1.9356 | Val AUC: 0.8527
Stage 2 Epoch 6/10 | Train Loss: 0.0298 | Val Loss: 1.9150 | Val AUC: 0.8319
Stage 2 Epoch 7/10 | Train Loss: 0.2136 | Val Loss: 1.7510 | Val AUC: 0.8840
Stage 2 Epoch 8/10 | Train Loss: 0.4515 | Val Loss: 1.0211 | Val AUC: 0.8710
Stage 2 Epoch 9/10 | Train Loss: 0.0758 | Val Loss: 1.3564 | Val AUC: 0.8492
Stage 2 Epoch 10/10 | Train Loss: 0.0268 | Val Loss: 1.5551 | Val AUC: 0.8877

Best Stage 2 validation AUC: 0.8877


### 5.4 Restore and save best Stage 2 checkpoint

In [13]:
# Restore the best Stage 2 checkpoint
model.load_state_dict(stage2_best_model_state)

# Save checkpoint (for use in other notebooks)
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()},
    "best_epoch": stage2_best_epoch,
    "best_val_auc": best_val_auc,
    "pos_weight": pos_weight.item(),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "stage1_epochs": STAGE1_EPOCHS,
    "stage2_epochs": STAGE2_EPOCHS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "history": history}

checkpoint_path = MODEL_DIR / "resnet18_best.pth"
torch.save(checkpoint, checkpoint_path)

print(f"Best epoch: {checkpoint['best_epoch']}")
print(f"Best validation AUC: {best_val_auc:.4f}")
print(f"Saved to: {checkpoint_path}")

Best epoch: 10
Best validation AUC: 0.8877
Saved to: /Users/hweeenheah/Downloads/EDA_Resources/nih-chestxray-ml/transfer-learning-resnet/models/resnet18_best.pth


In [14]:
# Verify saved checkpoint
saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False)

print("Checkpoint verification")
print("Best epoch:", saved_checkpoint["best_epoch"])
print("Best validation AUC:", saved_checkpoint["best_val_auc"])
print("Training history keys:", saved_checkpoint["history"].keys())
print("Total recorded epochs:", len(saved_checkpoint["history"]["train_loss"]))

Checkpoint verification
Best epoch: 10
Best validation AUC: 0.887678528201404
Training history keys: dict_keys(['train_loss', 'val_loss', 'val_auc'])
Total recorded epochs: 13


## 06. ImageNet-pretrained ResNet18 Training Summary

An ImageNet-pretrained ResNet18 was fine-tuned for binary classification of Effusion-positive and No Finding chest X-ray images.

#### Training Strategy
- **Stage 1 - Feature extraction:** The pretrained backbone was frozen while the new classification head was trained for 3 epochs.
- **Stage 2 - Fine-tuning:** All model parameters were unfrozen and trained for 10 additional epochs, using differential learning rates of 0.0001 for the backbone and 0.001 for the classification head.
- **Loss function:** BCEWithLogitsLoss with positive class weighting (7.2082).
- **Model selection:** Best checkpoint selected using validation ROC-AUC.

#### Results
- **Best Stage 1 validation ROC-AUC:** 0.8345
- **Best Stage 2 validation ROC-AUC:** 0.8877 (Epoch 10)
- **Final checkpoint:** 'models/resnet18_best.pth'

#### Observations

Fine-tuning improved validation ROC-AUC substantially compared with training only the classification head. However, training loss decreased sharply while validation loss fluctuated, suggesting overfitting during Stage 2.

The best validation ROC-AUC checkpoint was retained for subsequent evaluation. The held-out test set was not used during training or model selection. 